# HomeGuard – Smart Home Alarm Simulator
**Student:** Sergii Rostovskyi  
**Lab:** Programming Foundation Exercises

A simulated four-sensor home-monitoring system demonstrating dictionaries, `if/else`, functions, a `Sensor` class, alerts, and logging. **Run All** to reproduce the deterministic evidence and PNG files. No external sensors or APIs are used.

## Step 1 — Project setup
Import Python modules, define a reproducible random seed, and prepare simulated timestamps. The program advances by one *simulated* minute per iteration (no waiting).

In [1]:
import random
from datetime import datetime, timedelta
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt

random.seed(42)
BASE_TIME = datetime(2026, 10, 9, 14, 30, 0)
EVIDENCE_DIR = Path("homeguard_evidence")
EVIDENCE_DIR.mkdir(exist_ok=True)
print("Step 1: imports and evidence directory ready")

Step 1: imports and evidence directory ready


## Step 2 — Sensor, alert, and system-state dictionaries
Every sensor begins with `current_value=None`. An alert is a dictionary containing `time`, `severity`, `category`, and `message`.

In [2]:
def create_sensors():
    return [
        {"id": 1, "location": "Living Room", "type": "motion", "current_value": None},
        {"id": 2, "location": "Front Door", "type": "door", "current_value": None},
        {"id": 3, "location": "Kitchen", "type": "temperature", "current_value": None},
        {"id": 4, "location": "Bedroom", "type": "smoke", "current_value": None},
    ]

system_mode = "AWAY"
system_state = {"mode": system_mode, "alerts": [], "logs": []}
sensor_dicts = create_sensors()
assert len(sensor_dicts) == 4
assert all(sensor["current_value"] is None for sensor in sensor_dicts)
for sensor in sensor_dicts:
    print(sensor)

{'id': 1, 'location': 'Living Room', 'type': 'motion', 'current_value': None}
{'id': 2, 'location': 'Front Door', 'type': 'door', 'current_value': None}
{'id': 3, 'location': 'Kitchen', 'type': 'temperature', 'current_value': None}
{'id': 4, 'location': 'Bedroom', 'type': 'smoke', 'current_value': None}


## Step 3 — `if/else` decision logic
The same motion or door event only produces a security alert in AWAY mode. Smoke and dangerous temperatures produce safety alerts regardless of mode; comfort notices apply only in HOME mode. All boundaries are explicit.

In [3]:
FREEZE_TEMP = 35
OVERHEAT_TEMP = 95
COMFORT_MIN = 65
COMFORT_MAX = 75
VALID_MODES = {"HOME", "AWAY"}

def check_reading(sensor, mode, time_label="14:30:00"):
    if mode not in VALID_MODES:
        raise ValueError(f"Unsupported mode: {mode}")
    kind, value, location = sensor["type"], sensor["current_value"], sensor["location"]
    if value is None:
        return None  # Not yet read; prevents accidental false alerts.

    severity = category = message = None
    if kind == "smoke":
        if value:
            severity, category, message = "CRITICAL", "SAFETY", f"Smoke detected in {location}!"
    elif kind == "temperature":
        if value < FREEZE_TEMP:
            severity, category, message = "HIGH", "SAFETY", f"Temperature {value}°F in {location}: frozen pipe risk!"
        elif value > OVERHEAT_TEMP:
            severity, category, message = "HIGH", "SAFETY", f"Temperature {value}°F in {location}: overheating risk!"
        elif mode == "HOME" and (value < COMFORT_MIN or value > COMFORT_MAX):
            severity, category, message = "LOW", "COMFORT", f"Temperature {value}°F in {location} is outside 65–75°F."
    elif kind == "motion":
        if value and mode == "AWAY":
            severity, category, message = "HIGH", "SECURITY", f"Motion detected in {location} while in AWAY mode!"
    elif kind == "door":
        if value and mode == "AWAY":
            severity, category, message = "HIGH", "SECURITY", f"{location} opened while in AWAY mode!"
    else:
        raise ValueError(f"Unsupported sensor type: {kind}")

    if severity is None:
        return None
    return {"time": time_label, "severity": severity, "category": category, "message": message}

# Required hand-written example: same door reading, two different modes.
door_example = sensor_dicts[1].copy()
door_example["current_value"] = True
assert check_reading(door_example, "AWAY")["category"] == "SECURITY"
assert check_reading(door_example, "HOME") is None
print("AWAY door test:", check_reading(door_example, "AWAY"))
print("HOME door test:", check_reading(door_example, "HOME"))

AWAY door test: {'time': '14:30:00', 'severity': 'HIGH', 'category': 'SECURITY', 'message': 'Front Door opened while in AWAY mode!'}
HOME door test: None


## Step 4 — Small reusable functions
`generate_reading` uses random sensor readings. `process_readings` checks each one; `trigger_alert` displays the alert; `log_event` saves and displays notifications. A separate formatter keeps printed output consistent.

In [4]:
def generate_reading(sensor):
    kind = sensor["type"]
    if kind == "motion":
        value = random.random() < 0.35
    elif kind == "door":
        value = random.random() < 0.25
    elif kind == "temperature":
        value = random.randint(30, 100)
    elif kind == "smoke":
        value = random.random() < 0.15
    else:
        raise ValueError(f"Unsupported sensor type: {kind}")
    sensor["current_value"] = value
    return value

def format_reading(sensor):
    kind, value, location = sensor["type"], sensor["current_value"], sensor["location"]
    if value is None:
        return f"[READING] {location}: NOT READ"
    if kind == "motion":
        return f"[READING] {location} Motion: {'DETECTED' if value else 'No activity'}"
    if kind == "door":
        return f"[READING] {location}: {'OPENED' if value else 'CLOSED'}"
    if kind == "temperature":
        status = 'Danger' if value < FREEZE_TEMP or value > OVERHEAT_TEMP else ('Comfort range' if COMFORT_MIN <= value <= COMFORT_MAX else 'Outside comfort range')
        return f"[READING] {location} Temperature: {value}°F ({status})"
    if kind == "smoke":
        return f"[READING] {location} Smoke: {'DETECTED' if value else 'CLEAR'}"
    raise ValueError(kind)

def trigger_alert(alert, output_lines):
    line = f"[ALERT!] {alert['severity']}: {alert['category']}: {alert['message']}"
    output_lines.append(line)
    print(line)

def log_event(alert, logs, output_lines):
    line = f"[LOG] [{alert['time']}] Sending notification to homeowner..."
    logs.append({"time": alert["time"], "event": line, "alert": alert.copy()})
    output_lines.append(line)
    print(line)

def process_readings(sensors, mode, time_label, alerts, logs, output_lines):
    for sensor in sensors:
        generate_reading(sensor)
        line = format_reading(sensor)
        output_lines.append(line)
        print(line)
        alert = check_reading(sensor, mode, time_label)
        if alert:
            alerts.append(alert)
            trigger_alert(alert, output_lines)
            log_event(alert, logs, output_lines)

# Single-pass demonstration in two modes (independent of later seeded rounds).
for demo_mode in ("AWAY", "HOME"):
    demo_sensors = create_sensors()
    demo_alerts, demo_logs, demo_lines = [], [], []
    print(f"\nStep 4 smoke test, mode={demo_mode}")
    # Force the same known reading to make comparison meaningful.
    demo_sensors[1]["current_value"] = True
    demo_alert = check_reading(demo_sensors[1], demo_mode)
    if demo_alert:
        trigger_alert(demo_alert, demo_lines)
        log_event(demo_alert, demo_logs, demo_lines)
    else:
        print("No security alert (HOME mode)")


Step 4 smoke test, mode=AWAY
[ALERT!] HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:30:00] Sending notification to homeowner...

Step 4 smoke test, mode=HOME
No security alert (HOME mode)


## Step 5 — `Sensor` class
The class wraps sensor data and functionality. `read()` updates the reading, `isAbnormal()` delegates to the shared decision function, and `reset()` sets the value back to `None`. The simulator below uses **Sensor objects**, not the original dictionaries.

In [5]:
class Sensor:
    def __init__(self, sensor_id, location, sensor_type):
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.current_value = None

    def as_dict(self):
        return {"id": self.id, "location": self.location, "type": self.type, "current_value": self.current_value}

    def read(self):
        reading = self.as_dict()
        generate_reading(reading)
        self.current_value = reading["current_value"]
        return self.current_value

    def isAbnormal(self, mode, time_label="14:30:00"):
        return check_reading(self.as_dict(), mode, time_label)

    def reset(self):
        self.current_value = None

def create_sensor_objects():
    return [Sensor(s["id"], s["location"], s["type"]) for s in create_sensors()]

example_sensor = Sensor(1, "Living Room", "motion")
print("Before read:", example_sensor.current_value)
print("After read:", example_sensor.read())
example_sensor.reset()
assert example_sensor.current_value is None
print("After reset:", example_sensor.current_value)

def process_sensor_objects(sensors, mode, time_label, alerts, logs, output_lines):
    for sensor in sensors:
        sensor.read()
        line = format_reading(sensor.as_dict())
        output_lines.append(line)
        print(line)
        alert = sensor.isAbnormal(mode, time_label)
        if alert:
            alerts.append(alert)
            trigger_alert(alert, output_lines)
            log_event(alert, logs, output_lines)

Before read: None
After read: False
After reset: None


## Step 6 — Full integration: three simulated minutes and evidence PNGs
Each round uses four `Sensor` objects. Output is printed and preserved in `homeguard_evidence/iteration_01.png`, `iteration_02.png`, and `iteration_03.png`, plus `simulation_all_rounds.png`. The images are rendered directly from the captured program output, **not manually captured screenshots**.

In [6]:
def save_text_png(lines, file_path, title):
    # Render the program's actual console lines as an evidence image.
    fig_height = max(3.0, 0.31 * (len(lines) + 3))
    fig, ax = plt.subplots(figsize=(12.5, fig_height), dpi=150)
    fig.patch.set_facecolor("#101826")
    ax.set_facecolor("#101826")
    ax.axis("off")
    ax.text(0.025, 0.97, title, va="top", fontsize=15, weight="bold", color="#dbeafe", transform=ax.transAxes)
    for index, line in enumerate(lines):
        y = 0.89 - index * (0.79 / max(1, len(lines)))
        color = "#fda4af" if line.startswith("[ALERT!]") else ("#86efac" if line.startswith("[LOG]") else "#e2e8f0")
        ax.text(0.025, y, line, fontsize=10, fontfamily="DejaVu Sans Mono", color=color, va="top", transform=ax.transAxes)
    fig.savefig(file_path, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)

def run_simulation(mode="AWAY", rounds=3, seed=42):
    if mode not in VALID_MODES:
        raise ValueError(f"Unsupported mode: {mode}")
    random.seed(seed)
    sensors = create_sensor_objects()
    alerts, logs, all_lines, round_outputs = [], [], [], []
    header = "=== HomeGuard Security System ==="
    print(header)
    all_lines.append(header)
    for iteration in range(rounds):
        timestamp = (BASE_TIME + timedelta(minutes=iteration)).strftime("%H:%M:%S")
        lines = [f"Time: {timestamp}", f"Mode: {mode}"]
        print(f"\nTime: {timestamp}\nMode: {mode}")
        process_sensor_objects(sensors, mode, timestamp, alerts, logs, lines)
        round_outputs.append(lines)
        all_lines.extend(["", *lines])
        if mode == "AWAY":
            save_text_png(lines, EVIDENCE_DIR / f"iteration_{iteration + 1:02d}.png", f"HomeGuard – Iteration {iteration + 1}")
    print(f"\nSUMMARY: {rounds} rounds; {len(alerts)} alerts; {len(logs)} notifications")
    if mode == "AWAY":
        save_text_png(all_lines, EVIDENCE_DIR / "simulation_all_rounds.png", "HomeGuard – Full 3-round simulation")
    return {"mode": mode, "rounds": round_outputs, "alerts": alerts, "logs": logs, "lines": all_lines}

away_result = run_simulation("AWAY", rounds=3, seed=42)
assert len(away_result["rounds"]) == 3
assert len(away_result["alerts"]) >= 1, "The selected seed must give at least one visible alert"
assert len(away_result["logs"]) == len(away_result["alerts"])
print("\nGenerated evidence:")
for path in sorted(EVIDENCE_DIR.glob("*.png")):
    print(path)

=== HomeGuard Security System ===

Time: 14:30:00
Mode: AWAY
[READING] Living Room Motion: No activity
[READING] Front Door: OPENED
[ALERT!] HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:30:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 65°F (Comfort range)
[READING] Bedroom Smoke: CLEAR

Time: 14:31:00
Mode: AWAY
[READING] Living Room Motion: DETECTED
[ALERT!] HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [14:31:00] Sending notification to homeowner...
[READING] Front Door: OPENED
[ALERT!] HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:31:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 99°F (Danger)
[ALERT!] HIGH: SAFETY: Temperature 99°F in Kitchen: overheating risk!
[LOG] [14:31:00] Sending notification to homeowner...
[READING] Bedroom Smoke: DETECTED
[ALERT!] CRITICAL: SAFETY: Smoke detected in Bedroom!
[LOG] [14:31:00] Sending notification to homeowner...



Time: 14:32:00
Mode: AWAY
[READING] Living Room Motion: No activity
[READING] Front Door: OPENED
[ALERT!] HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:32:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 57°F (Outside comfort range)
[READING] Bedroom Smoke: CLEAR

SUMMARY: 3 rounds; 6 alerts; 6 notifications



Generated evidence:
homeguard_evidence/iteration_01.png
homeguard_evidence/iteration_02.png
homeguard_evidence/iteration_03.png
homeguard_evidence/simulation_all_rounds.png


## Step 7 — Deterministic tests and edge cases
The tests explicitly cover security, safety, comfort, the equality of threshold boundaries, missing readings, invalid modes, and an AWAY vs HOME simulation. Fixed values are deliberately used here so each expected result can be checked reliably.

In [7]:
def fixed_case(sensor_type, value, mode, expected_category, location="Test Room"):
    sensor = {"id": 99, "location": location, "type": sensor_type, "current_value": value}
    result = check_reading(sensor, mode)
    actual = result["category"] if result else None
    assert actual == expected_category, f"{sensor_type}={value} mode={mode}: expected {expected_category}, got {actual}"
    return f"PASS {sensor_type}={value}, mode={mode}: {actual or 'no alert'}"

fixed_tests = [
    ("motion", True, "AWAY", "SECURITY"),
    ("door", True, "AWAY", "SECURITY"),
    ("door", True, "HOME", None),
    ("motion", True, "HOME", None),
    ("smoke", True, "HOME", "SAFETY"),
    ("smoke", True, "AWAY", "SAFETY"),
    ("temperature", 34, "AWAY", "SAFETY"),
    ("temperature", 96, "HOME", "SAFETY"),
    ("temperature", 60, "HOME", "COMFORT"),
    ("temperature", 80, "AWAY", None),
    ("temperature", 65, "HOME", None),
    ("temperature", 75, "HOME", None),
    ("temperature", 35, "AWAY", None),
    ("temperature", 95, "AWAY", None),
    ("temperature", None, "AWAY", None),
]
test_lines = [fixed_case(*case) for case in fixed_tests]
for line in test_lines:
    print(line)
try:
    check_reading({"id": 1, "location": "Front Door", "type": "door", "current_value": True}, "INVALID")
except ValueError:
    test_lines.append("PASS invalid mode rejected with ValueError")
else:
    raise AssertionError("Invalid system mode should be rejected")
print(test_lines[-1])
save_text_png(test_lines, EVIDENCE_DIR / "fixed_tests.png", "HomeGuard – 16 fixed test checks")

print("\nHOME mode comparison (3 rounds):")
home_result = run_simulation("HOME", rounds=3, seed=42)
assert all(alert["category"] != "SECURITY" for alert in home_result["alerts"])
print("All tests passed.")

PASS motion=True, mode=AWAY: SECURITY
PASS door=True, mode=AWAY: SECURITY
PASS door=True, mode=HOME: no alert
PASS motion=True, mode=HOME: no alert
PASS smoke=True, mode=HOME: SAFETY
PASS smoke=True, mode=AWAY: SAFETY
PASS temperature=34, mode=AWAY: SAFETY
PASS temperature=96, mode=HOME: SAFETY
PASS temperature=60, mode=HOME: COMFORT
PASS temperature=80, mode=AWAY: no alert
PASS temperature=65, mode=HOME: no alert
PASS temperature=75, mode=HOME: no alert
PASS temperature=35, mode=AWAY: no alert
PASS temperature=95, mode=AWAY: no alert
PASS temperature=None, mode=AWAY: no alert
PASS invalid mode rejected with ValueError

HOME mode comparison (3 rounds):
=== HomeGuard Security System ===

Time: 14:30:00
Mode: HOME
[READING] Living Room Motion: No activity
[READING] Front Door: OPENED
[READING] Kitchen Temperature: 65°F (Comfort range)
[READING] Bedroom Smoke: CLEAR

Time: 14:31:00
Mode: HOME
[READING] Living Room Motion: DETECTED
[READING] Front Door: OPENED
[READING] Kitchen Temperature

## Brief explanation (2–3 sentences)
The simulator generates reproducible readings for motion, door, temperature, and smoke sensors and evaluates them against safety thresholds and HOME/AWAY mode rules. A `Sensor` class manages each reading, while helper functions display alerts, save notifications, and run three simulated minutes. The notebook also tests fixed edge cases and automatically exports PNG evidence of the actual output.

**How to run:** In Jupyter select the Python kernel, then choose **Run All**. PNG evidence will appear in the `homeguard_evidence` directory next to the notebook.

**Design choice / edge case:** The 35°F and 95°F boundaries are treated as safe endpoints: a temperature becomes a safety alert only *below* 35°F or *above* 95°F, matching the lab wording. A sensor with `None` is ignored until it has a valid reading.